### Tasks

* 1. Install TensorFlow and Keras in your Python environment, then write a script to create a simple 2-layer neural network (input, hidden, output) using Keras' Sequential API.

In [79]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [80]:
model = Sequential([
    Dense(units=8, activation='relu', input_shape=(4,), name='hidden_layer'),
    Dense(units=1, activation='sigmoid', name='output_layer')
])

model.summary()

Model: "sequential_24"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_layer (Dense)            │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49 (196.00 B)

 Trainable params: 49 (196.00 B)

 Non-trainable params: 0 (0.00 B)

* 2. Given a simple neural network with one hidden layer (using Keras), print out the weights before and after one training step on dummy data to observe how the weight update rule works.

In [81]:
from tensorflow.keras.optimizers import SGD

In [82]:
tf.random.set_seed(42)

model = Sequential([
    Dense(units=2, activation='sigmoid', input_shape=(2,), name='layer1')
])

In [83]:
model.compile(
    optimizer=SGD(learning_rate=0.5), 
    loss='mse'
)

In [84]:
X_dummy = np.array([[1.0, 2.0]])
y_dummy = np.array([[1.0]])

In [85]:
weights_before = model.get_weights()
print("* Weights BEFORE Training Step:")
print("Kernel (W):\n", weights_before[0])
print("Bias (b):  \n", weights_before[1])

* Weights BEFORE Training Step:
Kernel (W):
 [[ 0.6534221  -1.1348346 ]
 [ 0.51355135 -0.7323438 ]]
Bias (b):  
 [0. 0.]


In [86]:
model.train_on_batch(X_dummy, y_dummy)

weights_after = model.get_weights()
print("\n* Weights AFTER Training Step:")
print("Kernel (W):\n", weights_after[0])
print("Bias (b):  \n", weights_after[1])


* Weights AFTER Training Step:
Kernel (W):
 [[ 0.6638148  -1.104869  ]
 [ 0.5343367  -0.67241246]]
Bias (b):  
 [0.01039267 0.02996568]


* 3. Manually calculate the gradient of the loss with respect to the weights for a single neuron using the chain rule, given these values: input x=2, weight w=0.5, bias b=1, activation=sigmoid, target y=1.<br><br><em><strong>Hint:</strong> Write out each step of the derivative using the chain rule, then compute the numeric value.</em>

**Given Values:**
* Input x = 2
* Weight w = 0.5
* Bias b = 1
* Target y = 1
* Loss Function: Squared Error $L = \frac{1}{2} (a - y)^2$

**Forward Pass Computation**

Linear Combination:$$z = w \cdot x + b = (0.5 \cdot 2) + 1 = 2.0$$Sigmoid Activation:$$a = \sigma(z) = \frac{1}{1 + e^{-2.0}} \approx 0.8808$$Loss:$$L = \frac{1}{2}(0.8808 - 1.0)^2 \approx 0.0071

**Backpropagation via Chain Rule**

To find the gradient of the loss with respect to the weight, apply the chain rule:$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w}$$Step 1:$$\frac{\partial L}{\partial a} = a - y = 0.8808 - 1.0 = -0.1192$$Step 2:$$\frac{\partial a}{\partial z} = \sigma(z)(1 - \sigma(z)) = 0.8808 \cdot (1 - 0.8808) \approx 0.1049$$Step 3:$$\frac{\partial z}{\partial w} = x = 2.0$$

**Numeric Gradient**
$$\frac{\partial L}{\partial w} = (-0.1192) \cdot (0.1049) \cdot (2.0) \approx -0.0250$$

* 4. Build a function in Python that simulates vanishing gradient by applying the sigmoid activation repeatedly to an input (e.g., 0.9) for 10 layers, and print the final output.<br><br><em><strong>Hint:</strong> Observe how the value shrinks as you increase the number of layers.</em>

In [87]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def simulate_vanishing_gradients(x_init, num_layers=10):
    val = x_init
    print(f"Layer 0 (Input): {val:.6f}")
    
    for i in range(1, num_layers + 1):
        val = sigmoid(val)
        print(f"Layer {i:2d} Output: {val:.6f}")
        
    return val

final_val = simulate_vanishing_gradients(x_init=0.9, num_layers=10)

Layer 0 (Input): 0.900000
Layer  1 Output: 0.710950
Layer  2 Output: 0.670611
Layer  3 Output: 0.661640
Layer  4 Output: 0.659629
Layer  5 Output: 0.659177
Layer  6 Output: 0.659075
Layer  7 Output: 0.659053
Layer  8 Output: 0.659048
Layer  9 Output: 0.659046
Layer 10 Output: 0.659046


* 5. Use ChatGPT or Copilot to generate Python code that demonstrates exploding gradients in a deep network by initializing weights with large values, then explain in 2-3 lines what happens to the gradients during backpropagation.

In [88]:
from tensorflow.keras.initializers import Constant

In [89]:
model = Sequential()
model.add(Dense(10, input_shape=(10,), activation='linear', kernel_initializer=Constant(5.0)))

In [90]:
for _ in range(15):
    model.add(Dense(10, activation='linear', kernel_initializer=Constant(5.0)))

In [91]:
model.add(Dense(1, activation='linear', kernel_initializer=Constant(5.0)))

In [92]:
x = tf.ones((1, 10))
y = tf.constant([[1.0]])

In [93]:
with tf.GradientTape() as tape:
    predictions = model(x)
    loss = tf.keras.losses.mse(y, predictions)

gradients = tape.gradient(loss, model.trainable_variables)

print("First layer weight gradient norm:", tf.norm(gradients[0]).numpy())

First layer weight gradient norm: inf
